In [ ]:
!git clone https://github.com/seznam/MLPrague-2026.git
!pip install ./MLPrague-2026
!cp -a MLPrague-2026/images .
!cp -a MLPrague-2026/data .

# Introduction

In this notebook you will become familiar with the **YelpChi** dataset.
The dataset consists of real-world hotel and restaurant reviews, some of which have been marked as not recommended (i.e. spam) by Yelp.

We will perform:
* Explore the YelpChi dataset
* Review metrics that will be used for evaluation of a spam review detector
* Train a random as well as an informed non-graph spam detectors

In [ ]:
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rich import print
from rich.text import Text
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

from ml_prague_2026.evaluation import compare_models, evaluate_model

In [ ]:
YELP_CHI_PATH = "data/yelpchi.parquet"
YELP_CHI_FEATURES_PATH = "data/yelpchi_features.csv"

EXAMPLES_NORMAL = [7696, 37814, 12186]
EXAMPLES_SPAM = [44708, 43100, 42660]

---
## 1. Dataset

About **YelpChi:**
* Collected from Yelp.com and first used by Mukherjee et al. [1]
* Originally included 67,395 reviews for a set of hotels and restaurants (201) in the Chicago area
* Filtered to 45,954 reviews by Dou et al. [2] for the purpose of graph anomally detection benchmarking
* Reviews include product and user information, timestamp, ratings, and a plaintext review
* Yelp has a filtering algorithm in place that identifies fake/suspicious reviews and separates them into a list of **not recommended** items - marked as **spam** in the data
* Our task is to isolate the spam reviews (i.e. anomalies) from the other benign/normal posts

[1] Mukherjee, Arjun, et al. ["Fake review detection: Classification and analysis of real and pseudo reviews."](https://www2.cs.uh.edu/~arjun/papers/UIC-CS-TR-yelp-spam.pdf) UIC-CS-03-2013. Technical Report. 2013.  
[2] Dou, Yingtong, et al. ["Enhancing graph neural network-based fraud detectors against camouflaged fraudsters."](https://arxiv.org/pdf/2008.08692) Proceedings of the 29th ACM international conference on information & knowledge management. 2020.

### Example - Product reviews

![yelp-chi-reviews.png](images/yelp-chi-reviews.png)

### Example - Not recommended reviews

![yelp-chi-not-recommended.png](images/yelp-chi-not-recommended.png)

### Data

In [ ]:
yelp_chi = pd.read_parquet(YELP_CHI_PATH)

In [ ]:
yelp_chi.columns

In [ ]:
FEATURES = pd.read_csv(YELP_CHI_FEATURES_PATH, index_col=0, header=None, names=["feature", "name"])["name"].to_dict()

In [ ]:
yelp_chi[[c for c in yelp_chi.columns if c not in FEATURES.keys()]].head(3)

### Benign examples

In [ ]:
for i, r in enumerate(yelp_chi["review"][EXAMPLES_NORMAL]):
    print(f"[#008f39]\u2705 Example #{i + 1}:[/#008f39]")
    print(Text(textwrap.fill(r, 80) + "\n\n"))

### Spam examples

In [ ]:
for i, r in enumerate(yelp_chi["review"][EXAMPLES_SPAM]):
    print(f"[#d32f2f]\u274c Example #{i + 1}:[/#d32f2f]")
    print(Text(textwrap.fill(r, 80) + "\n\n"))

### Review statistics

### TASK 1.A: Spam reviews

What is the ratio of spam reviews?

Run the following cell.

In [ ]:
yelp_chi["spam"].value_counts() / len(yelp_chi)

### User behaviour

### TASK 1.B: User count

What is the number of users?

Run the following cell.

In [ ]:
yelp_chi["user_id"].nunique()

### TASK 1.C: Spam users

What is the ratio of users producing spam reviews?

Mark users who produced at least 1 spam review using a new column `user_spam`.

In [ ]:
# TASK 1.C: Spam users
# - Mark users who produced at least one spam review using a new column "user_spam"
#
# - Hint: You can `groupby` by "user_id" and `transform` the "spam" column using the "any" function

yelp_chi["user_spam"] = None ### YOUR CODE HERE ###

yelp_chi.drop_duplicates("user_id")["user_spam"].value_counts() / yelp_chi["user_id"].nunique()

<!-- #region -->
<details>
<summary><b>Task 1.C — Solution</b></summary>

```python
yelp_chi["user_spam"] = yelp_chi.groupby("user_id")["spam"].transform("any")
```

</details>
<!-- #endregion -->

### TASK 1.D: Spam users review count

Do behaviour of spam users differ from normal users? Consider review count.

Run the following cell.

In [ ]:
def compare_feature(df, feature, user_spam_col="user_spam", figsize=(10, 3)):
    fig, ax = plt.subplots(1, 1, figsize=figsize)

    bin_edges = np.histogram_bin_edges(yelp_chi[feature], 60)

    yelp_chi[yelp_chi[user_spam_col] == False][feature].plot.hist(bins=bin_edges, density=True, histtype="step", ax=ax)
    yelp_chi[yelp_chi[user_spam_col] == True][feature].plot.hist(bins=bin_edges, density=True, histtype="step", ax=ax)

    plt.title(f"Distributions of {" ".join(feature.split("_")[1:])} for normal vs. spam users")
    plt.legend(["normal", "spam"])
    plt.show()


yelp_chi["user_review_count"] = yelp_chi.groupby("user_id").transform("size")

compare_feature(yelp_chi, "user_review_count")

### TASK 1.E: Spam users activity span

Do behaviour of spam users differ from normal users? Consider activity span.

Run the following cell.

In [ ]:
yelp_chi["user_activity_span"] = yelp_chi.groupby("user_id")["date"].transform(lambda x: x.max() - x.min()).dt.days

compare_feature(yelp_chi, "user_activity_span")

### Features

The dataset already provides hand-crafted features of the reviews. The features were obtained by extracting text features for reviews and combining them with behavioural features of the following entities:
1. Review - `f_r...` (15 features)
2. User - `f_u...` (7) features
3. Product (hotel/restaurant) - `f_p...` (8 features)

For more details, please refer to the work done by Rayana et al. [1].

[1] Rayana, Shebuti, and Leman Akoglu. ["Collective opinion spam detection: Bridging review networks and metadata."](https://shebuti.com/wp-content/uploads/2016/06/15-kdd-collectiveopinionspam.pdf) Proceedings of the 21th ACM SIGKDD international conference on knowledge discovery and data mining. 2015.

### TASK 1.F: Review node features

The feature `f_u_mnr` is the maximum number of reviews (MNR) posted by a user in a day.
**Warning**, the feature was normalized to `[0, 1]` in a non-linear way and 0 means the most extreme MNR value.  
What is the ratio of reviews written by users with high MNR?

Run the following cell.

In [ ]:
yelp_chi["f_u_mnr"].plot.hist(bins=10, title="Histogram of MNR (maximum number of reviews in a day by user) for reviews", figsize=(6, 3));

---
## 2. Anomaly detection

**Classification** metrics:
* *precision* - the accuracy of positive predictions
* *recall* - the ability to find all actual positives
* easy to interpret
* we compute macro average of metrics for normal and spam users

**Ranking** metrics:
* AUPRC - area under *precision-recall* curve, computed as [*average precision*](https://en.wikipedia.org/wiki/Evaluation_measures_(information_retrieval)#Average_precision)
* Rec@K - *recall* at K, where K = number of anomalies
* enable to evaluate models while not assuming a specific class thresholds as opposed to the classification ones

### Train/test datasets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(yelp_chi[FEATURES.keys()], yelp_chi["spam"].to_numpy(), test_size=0.2, stratify=yelp_chi["spam"])

### Random baseline

### TASK 2.A: Train a random model

Train a dummy, i.e., random, classifier and review the metrics.

Run the following cell.

In [ ]:
clf_base = DummyClassifier(strategy="stratified")
clf_base.fit(X_train, y_train)

metrics_base = evaluate_model(
    "baseline",
    y_test,
    clf_base.predict(X_test),
    clf_base.predict_proba(X_test)[:, 1]
)

### Decision tree classifier

### TASK 2.B: Train an informed model

Train `DecisionTreeClassifier`model using the features `X_train` and the labels `y_train` and review the metrics.

In [ ]:
# TASK 2.B: train a baseline classifier
# - Init and train `DecisionTreeClassifier` model using `X_train` and `y_train`, limit `max_depth=2` and fix `random_state=42`

### YOUR CODE HERE ###

clf_dt = None

### YOUR CODE HERE ###

metrics_dt = evaluate_model(
    "decision_tree",
    y_test,
    clf_dt.predict(X_test),
    clf_dt.predict_proba(X_test)[:, 1]
)

<!-- #region -->
<details>
<summary><b>Task 2.B — Solution</b></summary>

```python
clf_dt = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_dt.fit(X_train, y_train)
```

</details>
<!-- #endregion -->

### TASK 2.C: Explore important review features

The following features were selected as most important by the model:
* ISR - Whether a review is a single review (of a user)
* ETG (u) - Entropy of temporal gaps (of review's user)
* Rank - Chronological order among all the reviews (of a product)

Run the following cell.

In [ ]:
plt.figure(figsize=(10, 5))
_ = plot_tree(clf_dt, feature_names=list(FEATURES.values()), class_names=["Normal", "Spam"], filled=True, fontsize=10)

### TASK 2.D: Review metrics

Compare the baseline models using the choosen metrics.

Run the following cell.

In [ ]:
compare_models([metrics_base, metrics_dt])